# 06 - Dataset Splitting

This notebook creates stratified training, validation, and testing datasets for the fraud detection system. The split preserves the original class distribution and prepares the data for model development while preventing data leakage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

## Load Dataset

Load the cleaned dataset and recreate the engineered features developed during the feature engineering phase.

In [2]:
DATA_PATH = Path("../data/raw/creditcard.csv")

df = pd.read_csv(DATA_PATH)

df = df.drop_duplicates().reset_index(drop=True)

## Recreate Engineered Features

Rebuild the engineered features so that the dataset matches the feature engineering phase.

In [3]:
df["TransactionHour"] = (df["Time"] // 3600).astype(int)

df["LogAmount"] = np.log1p(df["Amount"])

threshold = df["Amount"].quantile(0.95)

df["HighValueTransaction"] = (
    df["Amount"] > threshold
).astype(int)

## Separate Features and Target

Split the dataset into input features (`X`) and the target variable (`y`).

In [4]:
TARGET_COLUMN = "Class"

X = df.drop(columns=TARGET_COLUMN)

y = df[TARGET_COLUMN]

print(f"Features Shape : {X.shape}")
print(f"Target Shape   : {y.shape}")

Features Shape : (283726, 33)
Target Shape   : (283726,)


## Train / Validation / Test Split

Perform a two-stage stratified split:

- 80% Training
- 10% Validation
- 10% Testing

Stratification preserves the original fraud-to-normal transaction ratio across all datasets.

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

## Dataset Sizes

Review the number of samples in each dataset after splitting.

In [6]:
print(f"Training Samples   : {len(X_train):,}")
print(f"Validation Samples : {len(X_valid):,}")
print(f"Testing Samples    : {len(X_test):,}")

Training Samples   : 226,980
Validation Samples : 28,373
Testing Samples    : 28,373


## Split Summary

The dataset has been successfully divided into training, validation, and testing subsets using stratified sampling.

The preprocessing pipeline will be fitted **only on the training dataset** during the model development phase.

## 2.7.1 Train / Validation / Test Split

The cleaned dataset was divided into training (80%), validation (10%), and testing (10%) subsets using stratified sampling. A two-stage split ensured that the original class distribution was preserved across all datasets. The resulting datasets will be used throughout model training, hyperparameter tuning, and final evaluation while preventing data leakage.

In [7]:
def class_distribution(target):
    distribution = target.value_counts().sort_index()
    percentage = target.value_counts(normalize=True).sort_index() * 100

    return pd.DataFrame({
        "Count": distribution,
        "Percentage (%)": percentage.round(4)
    })

## Original Dataset Distribution

Review the class distribution in the complete dataset.

In [8]:
original_distribution = class_distribution(y)

original_distribution

,Count,Percentage (%)
Class,,
0,283253,99.8333
1,473,0.1667


## Training Dataset Distribution

Verify the class distribution in the training dataset.

In [9]:
train_distribution = class_distribution(y_train)

train_distribution

,Count,Percentage (%)
Class,,
0,226602,99.8335
1,378,0.1665


## Validation Dataset Distribution

Verify the class distribution in the validation dataset.

In [10]:
validation_distribution = class_distribution(y_valid)

validation_distribution

,Count,Percentage (%)
Class,,
0,28325,99.8308
1,48,0.1692


## Testing Dataset Distribution

Verify the class distribution in the testing dataset.

In [11]:
test_distribution = class_distribution(y_test)

test_distribution

,Count,Percentage (%)
Class,,
0,28326,99.8343
1,47,0.1657


## Distribution Comparison

Compare the percentage of fraudulent transactions across all dataset splits.

In [12]:
comparison_df = pd.DataFrame({
    "Original (%)": original_distribution["Percentage (%)"],
    "Train (%)": train_distribution["Percentage (%)"],
    "Validation (%)": validation_distribution["Percentage (%)"],
    "Test (%)": test_distribution["Percentage (%)"]
})

comparison_df

,Original (%),Train (%),Validation (%),Test (%)
Class,,,,
0,99.8333,99.8335,99.8308,99.8343
1,0.1667,0.1665,0.1692,0.1657


## Stratification Summary

The class distribution has been successfully preserved across the training, validation, and testing datasets using stratified sampling.

This ensures that each dataset accurately represents the original fraud-to-normal transaction ratio, reducing sampling bias and supporting reliable model evaluation.

## 2.7.2 Stratification Check

A stratification check was performed to verify that the class distribution remained consistent across the training, validation, and testing datasets. The comparison confirmed that the fraud-to-normal transaction ratio was preserved, ensuring representative datasets for model training and evaluation.

In [13]:
from pathlib import Path

OUTPUT_DIR = Path("../data/processed")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Save Feature Datasets

Store the feature datasets as CSV files.

In [14]:
X_train.to_csv(OUTPUT_DIR / "X_train.csv", index=False)
X_valid.to_csv(OUTPUT_DIR / "X_valid.csv", index=False)
X_test.to_csv(OUTPUT_DIR / "X_test.csv", index=False)

## Save Target Datasets

Store the target labels as CSV files.

In [15]:
y_train.to_frame().to_csv(
    OUTPUT_DIR / "y_train.csv",
    index=False
)

y_valid.to_frame().to_csv(
    OUTPUT_DIR / "y_valid.csv",
    index=False
)

y_test.to_frame().to_csv(
    OUTPUT_DIR / "y_test.csv",
    index=False
)

## Verify Saved Files

Confirm that all processed datasets have been saved successfully.

In [16]:
saved_files = sorted(OUTPUT_DIR.glob("*.csv"))

print("Saved Files:\n")

for file in saved_files:
    print(file.name)

Saved Files:

X_test.csv
X_train.csv
X_valid.csv
y_test.csv
y_train.csv
y_valid.csv


## Dataset Summary

Display the shape of each saved dataset.

In [17]:
summary_df = pd.DataFrame({
    "Dataset": [
        "X_train",
        "X_valid",
        "X_test",
        "y_train",
        "y_valid",
        "y_test"
    ],
    "Rows": [
        len(X_train),
        len(X_valid),
        len(X_test),
        len(y_train),
        len(y_valid),
        len(y_test)
    ],
    "Columns": [
        X_train.shape[1],
        X_valid.shape[1],
        X_test.shape[1],
        1,
        1,
        1
    ]
})

summary_df

,Dataset,Rows,Columns
0,X_train,226980,33
1,X_valid,28373,33
2,X_test,28373,33
3,y_train,226980,1
4,y_valid,28373,1
5,y_test,28373,1


## Save Summary

The processed datasets have been successfully saved.

These datasets will serve as the standardized inputs for model training, validation, testing, and deployment throughout the remainder of the project.